# Step 5 — Model Comparison

Project 2 (HPDP, SECP3133) — Real-Time Sentiment Analysis on Grab Google Play Reviews

**Owner:** Syahmi (NLP & Model Engineer) · **Purpose:** Side-by-side comparison of the three trained models and selection of the model to deploy in the Kafka + Spark pipeline (§6.3 of the brief).

This notebook is a reference for the **Sentiment Model Development** and **Optimization and Comparison** sections of the final report. Every table and chart below is ready to drop straight into the PDF.

Models compared:
1. **Naive Bayes** — TF-IDF + MultinomialNB (classical ML baseline)
2. **LSTM** — Bidirectional LSTM on tokenised sequences (deep learning)
3. **DistilBERT** — `distilbert-base-multilingual-cased` fine-tuned (transformer)

## 0. Setup

In [ ]:
import json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.image import imread

MODELS_DIR = '../models'
FIGS_DIR   = '../figures'
LABELS     = ['Negative', 'Neutral', 'Positive']

def load_metrics(name):
    with open(f'{MODELS_DIR}/metrics_{name}.json') as f:
        return json.load(f)

metrics = {
    'Naive Bayes': load_metrics('naive_bayes'),
    'LSTM'       : load_metrics('lstm'),
    'DistilBERT' : load_metrics('distilbert'),
}
print('Loaded metrics for:', list(metrics.keys()))

## 1. EDA recap (for the report's Data section)

Three figures produced in Step 1 (`01_eda_split.ipynb`) describe the dataset the models were trained on. They explain *why* the imbalance and short-text effects matter for the comparison that follows.

In [ ]:
eda_files = [
    ('eda_class_distribution.png', 'Class distribution — Negative dominates, Neutral is rare'),
    ('eda_token_lengths.png',      'Token-length distribution per class'),
    ('eda_wordclouds.png',         'Word clouds per sentiment class'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (fname, title) in zip(axes, eda_files):
    ax.imshow(imread(f'{FIGS_DIR}/{fname}'))
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Headline metrics table

Accuracy and per-class F1 on the held-out **test set** (4 500 reviews). Macro-F1 is the headline metric — it weights all three classes equally, which is the fair number for an imbalanced 3-class problem.

In [ ]:
rows = []
for name, m in metrics.items():
    rows.append({
        'Model'          : name,
        'Accuracy'       : m['accuracy'],
        'Macro-F1'       : m['macro_f1'],
        'Macro-Precision': m['macro_precision'],
        'Macro-Recall'   : m['macro_recall'],
        'F1 Negative'    : m['f1_negative'],
        'F1 Neutral'     : m['f1_neutral'],
        'F1 Positive'    : m['f1_positive'],
    })
headline = pd.DataFrame(rows).set_index('Model').round(4)
headline

**Reading the table.**

- LSTM wins on raw **accuracy** (0.845) — but only because it predicts the majority classes well and *completely ignores* Neutral (F1 = 0.000).
- DistilBERT wins on **Macro-F1** (0.637) and is the only model that classifies the Neutral class with non-trivial skill.
- Naive Bayes is competitive on Macro-F1 (0.623) despite being a 100x simpler model — the strong TF-IDF baseline that every classical-NLP paper warns you not to ignore.

## 3. Macro-F1 vs accuracy (bar chart)

The chart makes the LSTM accuracy/F1 gap visually obvious — useful for the report's Results section.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(headline.index))
w = 0.35
ax.bar(x - w/2, headline['Accuracy'], w, label='Accuracy',  color='#4C72B0')
ax.bar(x + w/2, headline['Macro-F1'], w, label='Macro-F1', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(headline.index)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Accuracy vs Macro-F1 (test set)')
for i, (a, f) in enumerate(zip(headline['Accuracy'], headline['Macro-F1'])):
    ax.text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, f + 0.01, f'{f:.3f}', ha='center', fontsize=9)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/compare_accuracy_vs_f1.png', dpi=120)
plt.show()

## 4. Per-class F1 (the neutral-class story)

Grouped bars by sentiment class — this is the single most important comparison chart because it exposes *which* class each model is failing on.

In [ ]:
per_class = headline[['F1 Negative', 'F1 Neutral', 'F1 Positive']].T
per_class.index = LABELS

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(LABELS))
w = 0.25
colors = {'Naive Bayes': '#55A868', 'LSTM': '#C44E52', 'DistilBERT': '#8172B2'}
for i, model in enumerate(per_class.columns):
    offset = (i - 1) * w
    bars = ax.bar(x + offset, per_class[model], w, label=model, color=colors[model])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1 score')
ax.set_title('Per-class F1 — Neutral exposes the real differences between models')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/compare_per_class_f1.png', dpi=120)
plt.show()

**Interpretation for the report.**

All three models handle the polar classes (Negative ≈ 0.81–0.88, Positive ≈ 0.84–0.87) well. The differentiator is the **Neutral** class:

| Model       | F1 Neutral | What it means                                              |
|-------------|------------|------------------------------------------------------------|
| LSTM        | 0.000      | Collapsed — never predicts Neutral. The accuracy is misleading. |
| Naive Bayes | 0.180      | Recognises Neutral occasionally; better than nothing.      |
| DistilBERT  | 0.207      | Best, thanks to multilingual pre-training + class-weighted loss. |

Neutral is hard because (a) it is the rarest class (~10 % of the data per Section 1) and (b) Neutral reviews share vocabulary with both extremes ("app okay tapi lambat").

## 5. Confusion matrices side-by-side

Reusing the PNGs each training notebook already produced. These are the canonical confusion-matrix figures for the report — no need to regenerate.

In [ ]:
cm_files = [
    ('nb_confusion_matrix.png',         'Naive Bayes'),
    ('lstm_confusion_matrix.png',       'LSTM'),
    ('distilbert_confusion_matrix.png', 'DistilBERT'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (fname, title) in zip(axes, cm_files):
    ax.imshow(imread(f'{FIGS_DIR}/{fname}'))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout(); plt.show()

**What the diagonals tell you.** All three matrices show the same pattern: the Neutral row is the weakest. LSTM has an essentially empty Neutral-predicted column (everything Neutral is mis-routed to Negative or Positive). DistilBERT distributes its Neutral predictions across the actual Neutral row more often than the other two.

## 6. Compute cost: train time & inference latency

Important context for the **Pipeline Integration** section. The latency number here drives whether the chosen model can keep up with the Kafka stream.

In [ ]:
cost = pd.DataFrame({
    'Train time (s)'      : [m['train_time_s']            for m in metrics.values()],
    'Inference (ms/sample)': [m['inference_ms_per_sample'] for m in metrics.values()],
    'Throughput (samples/s)': [1000 / m['inference_ms_per_sample'] for m in metrics.values()],
}, index=metrics.keys()).round(3)
cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(cost.index, cost['Train time (s)'], color='#4C72B0')
axes[0].set_yscale('log')
axes[0].set_ylabel('Train time (s, log scale)')
axes[0].set_title('Training cost')
for i, v in enumerate(cost['Train time (s)']):
    axes[0].text(i, v * 1.2, f'{v:.2f}s', ha='center', fontsize=9)

axes[1].bar(cost.index, cost['Inference (ms/sample)'], color='#DD8452')
axes[1].set_yscale('log')
axes[1].set_ylabel('Inference (ms/sample, log scale)')
axes[1].set_title('Inference latency')
for i, v in enumerate(cost['Inference (ms/sample)']):
    axes[1].text(i, v * 1.2, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/compare_compute_cost.png', dpi=120)
plt.show()

**Takeaways for streaming.**

- Naive Bayes is effectively free (~1 µs/review, ~1 M reviews/sec on a single core).
- LSTM is ~300x slower than NB but still well under 1 ms.
- DistilBERT is the slowest at ~1 ms/review = ~1 000 reviews/sec on the same GPU. Comfortable headroom for Grab Play Store review volume, but to be aware of when sizing the Spark executors.

## 7. Decision: which model to deploy

Selection criteria, in priority order:

1. **Macro-F1** — the fair metric on an imbalanced 3-class problem.
2. **Neutral-class F1** — the report needs a model that can actually distinguish all three classes.
3. **Inference latency** — must fit inside the Spark micro-batch budget.

| Criterion           | Winner       |
|---------------------|--------------|
| Macro-F1            | **DistilBERT** (0.637) |
| F1 Neutral          | **DistilBERT** (0.207) |
| Latency             | Naive Bayes (but DistilBERT at ~1 ms is still well within budget) |

**Selected model for the Kafka + Spark pipeline: DistilBERT.**

Naive Bayes is kept as a **fallback baseline** — useful for batch vs streaming benchmarking in Week 4 (Section 6.4 of the brief) because the speed gap will let us isolate model cost from pipeline overhead.

## 8. Export comparison table for the report

In [ ]:
combined = headline.join(cost)
combined.to_csv(f'{MODELS_DIR}/model_comparison.csv')
print('Saved:', f'{MODELS_DIR}/model_comparison.csv')
combined

---

**[OK] Step 5 complete.** Model comparison artefacts produced:

- `models/model_comparison.csv` — full numeric table
- `figures/compare_accuracy_vs_f1.png`
- `figures/compare_per_class_f1.png`
- `figures/compare_compute_cost.png`

Hand-off to **Pipeline & Visualization Engineer**: deploy `models/distilbert/` inside the Spark Structured Streaming job.